In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [2]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_21632/4196082941.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_21632/4196082941.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
    def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q_val),
                0, 1, n=n_points
            )[0]

    integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
    
    return integral_value

In [6]:
# def model function
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [7]:
# set cost and minimize
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.93 (χ²/ndof = 0.2)      │              Nfcn = 341              │
│ EDM = 3.1e-06 (Goal: 0.0002)     │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0616   │  0.0022   │            │            │         │         │       │
│ 1 │ mg   │   0.389   │   0.005   │            │            │         │         │       │
│ 2 │ a1   │   1.49    │   0.05    │            │            │         │         │       │
│ 3 │ a2   │   2.16    │   0.31    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 4.85e-06    10e-6    54e-6  -157e-6 │
│  mg │    10e-6 2.41e-05 0.056e-3 0.101e-3 │
│  a1 │    54e-6 0.056e-3   0.0023  -0.0136 │
│  a2 │  -157e-6 0.101e-3  -0.0136   0.0956 │
└─────┴─────────────────────────────────────┘

In [8]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001
    n_points = 10000

    # def integrand(y, x, mg, a1, a2, m2_func, q_val):
    #     k        = sqrt_s * x
    #     phi      = 2*np.pi*y
    #     jacobian = 2*np.pi*sqrt_s
    #     return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
    #               T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#-----------------------------------------------------------------------------------------------

#-----------------------------------------------------------------------------------------------

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': 'blue', 'width': 2},
              'marker': {'size': 4},
              'mode': 'lines+markers',
              'name': '7 TeV, pl',
              'showlegend': True,
              'type': 'scatter',
              'x': [0.006, 0.007, 0.008, 0.009000000000000001,
                    0.010000000000000002, 0.011000000000000003,
                    0.012000000000000004, 0.013000000000000005,
                    0.014000000000000005, 0.015000000000000006,
                    0.016000000000000007, 0.017000000000000008,
                    0.01800000000000001, 0.01900000000000001, 0.02000000000000001,
                    0.02100000000000001, 0.022000000000000013,
                    0.023000000000000013, 0.024000000000000014,
                    0.025000000000000015, 0.026000000000000016,
                    0.027000000000000017, 0.028000000000000018,
                    0.02900000000000002, 0.03000000000000002, 0.03100000000000002,
                    0.03200000000000002, 0.03300000000000002, 0.03400000000000002,
                    0.035000000000000024, 0.036000000000000025,
                    0.037000000000000026, 0.03800000000000003, 0.03900000000000003,
                    0.04000000000000003, 0.04100000000000003, 0.04200000000000003,
                    0.04300000000000003, 0.04400000000000003, 0.04500000000000003,
                    0.046000000000000034, 0.047000000000000035,
                    0.048000000000000036, 0.04900000000000004, 0.05000000000000004,
                    0.05100000000000004, 0.05200000000000004, 0.05300000000000004,
                    0.05400000000000004, 0.05500000000000004, 0.05600000000000004,
                    0.057000000000000044, 0.058000000000000045,
                    0.059000000000000045, 0.060000000000000046,
                    0.06100000000000005, 0.06200000000000005, 0.06300000000000004,
                    0.06400000000000004, 0.06500000000000004, 0.06600000000000004,
                    0.06700000000000005, 0.06800000000000005, 0.06900000000000005,
                    0.07000000000000005, 0.07100000000000005, 0.07200000000000005,
                    0.07300000000000005, 0.07400000000000005, 0.07500000000000005,
                    0.07600000000000005, 0.07700000000000005, 0.07800000000000006,
                    0.07900000000000006, 0.08000000000000006, 0.08100000000000006,
                    0.08200000000000006, 0.08300000000000006, 0.08400000000000006,
                    0.08500000000000006, 0.08600000000000006, 0.08700000000000006,
                    0.08800000000000006, 0.08900000000000007, 0.09000000000000007,
                    0.09100000000000007, 0.09200000000000007, 0.09300000000000007,
                    0.09400000000000007, 0.09500000000000007, 0.09600000000000007,
                    0.09700000000000007, 0.09800000000000007, 0.09900000000000007,
                    0.10000000000000007, 0.10100000000000008, 0.10200000000000008,
                    0.10300000000000008, 0.10400000000000008, 0.10500000000000008,
                    0.10600000000000008, 0.10700000000000008, 0.10800000000000008,
                    0.10900000000000008, 0.11000000000000008, 0.11100000000000008,
                    0.11200000000000009, 0.11300000000000009, 0.11400000000000009,
                    0.11500000000000009, 0.11600000000000009, 0.11700000000000009,
                    0.11800000000000009, 0.11900000000000009, 0.12000000000000009,
                    0.1210000000000001, 0.1220000000000001, 0.1230000000000001,
                    0.1240000000000001, 0.12500000000000008, 0.12600000000000008,
                    0.12700000000000009, 0.12800000000000009, 0.1290000000000001,
                    0.1300000000000001, 0.1310000000000001, 0.1320000000000001,
                    0.1330000000000001, 0.1340000000000001, 0.1350000000000001,
                    0.1360000000000001, 0.1370000000000001, 0.1380000000000001,
                    0.139000000000000

In [9]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:
        s = sqrt_s ** 2


        integral_value = full_int(mg, a1, a2, mg_model, 0.0, sqrt_s) 

        born_amp = amp_calculation(integral_value, s, epsilon, 0)
        lst_born_amp.append(born_amp)
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon, 0), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    minuit_born.values['eps'],
    minuit_born.values['mg'],
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

/tmp/ipykernel_21632/1760823969.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': 'blue', 'dash': 'solid', 'width': 2},
              'marker': {'size': 3},
              'mode': 'lines+markers',
              'name': 'PL Atlas',
              'showlegend': True,
              'type': 'scatter',
              'x': [1, 101, 201, 301, 401, 501, 601, 701, 801, 901, 1001, 1101,
                    1201, 1301, 1401, 1501, 1601, 1701, 1801, 1901, 2001, 2101,
                    2201, 2301, 2401, 2501, 2601, 2701, 2801, 2901, 3001, 3101,
                    3201, 3301, 3401, 3501, 3601, 3701, 3801, 3901, 4001, 4101,
                    4201, 4301, 4401, 4501, 4601, 4701, 4801, 4901, 5001, 5101,
                    5201, 5301, 5401, 5501, 5601, 5701, 5801, 5901, 6001, 6101,
                    6201, 6301, 6401, 6501, 6601, 6701, 6801, 6901, 7001, 7101,
                    7201, 7301, 7401, 7501, 7601, 7701, 7801, 7901, 8001, 8101,
                    8201, 8301, 8401, 8501, 8601, 8701, 8801, 8901, 9001, 9101,
                    9201, 9301, 9401, 9501, 9601, 9701, 9801, 9901, 10001, 10101,
                    10201, 10301, 10401, 10501, 10601, 10701, 10801, 10901, 11001,
                    11101, 11201, 11301, 11401, 11501, 11601, 11701, 11801, 11901,
                    12001, 12101, 12201, 12301, 12401, 12501, 12601, 12701, 12801,
                    12901, 13001],
              'y': [30.5327497097743, 56.878524005172665, 61.91015455624397,
                    65.06731721783062, 67.40740398684368, 69.28156103441187,
                    70.8521281788228, 72.2081560712469, 73.4040103181731,
                    74.47542718122267, 75.4472086444109, 76.33730099647279,
                    77.15912638037716, 77.92299676961382, 78.63701323734209,
                    79.30766070897727, 79.94021443726842, 80.53902566847117,
                    81.10772729491009, 81.64938504095024, 82.16661067431757,
                    82.66164817647548, 83.13644029337965, 83.59268061050835,
                    84.03185478508969, 84.45527354510237, 84.86409935851128,
                    85.25936818067794, 85.64200733474523, 86.01285032455394,
                    86.37264919275282, 86.72208489826718, 87.0617760835201,
                    87.39228652323835, 87.71413148663387, 88.02778319844575,
                    88.33367554832374, 88.63220816981202, 88.92374998791189,
                    89.20864231649057, 89.48720157262672, 89.75972166357315,
                    90.02647609277395, 90.28771982384595, 90.5436909352756,
                    90.7946120935103, 91.04069186793606, 91.2821259077522,
                    91.51909799785398, 91.75178100840321, 91.98033775072588,
                    92.20492175045112, 92.42567794734833, 92.64274333007705,
                    92.8562475130079, 93.06631326136973, 93.273056970199,
                    93.47658910190388, 93.67701458667666, 93.87443318949059,
                    94.06893984698591, 94.26062497717233, 94.44957476454792,
                    94.63587142294972, 94.81959343819727, 95.00081579237306,
                    95.17961017138865, 95.35604515731521, 95.53018640680546,
                    95.70209681680238, 95.87183667860978, 96.03946382129601,
                    96.20503374530857, 96.36859974709328, 96.5302130354383,
                    96.68992284019598, 96.84777651397651, 97.00381962735372,
                    97.15809605807519, 97.31064807472609, 97.46151641525694,
                    97.61074036075067, 97.75835780477215, 97.90440531861559,
                    98.04891821273837, 98.19193059464679, 98.33347542347815,
                    98.4735845615029, 98.61228882275502, 98.74961801898009,
                    98.88560100307916, 99.02026571020856, 99.15363919668934,
                    99.2857476768626, 99.41661655802258, 99.54627047354477,
                    99.67473331432191, 99.80202825861048, 99.92817780038277,
                    100.053203776275, 100.17712739121374, 100.29996924279773,
                    100.42174934450809, 100.542487147

In [10]:
def eik_amp(s, b, q, chi):
    return 1j * s * b * j0(b*q) * (1 - np.exp(1j*chi))

print(lst_born_amp)

[78.41389592681145j, 1490109.4873411304j, 6423638.869562708j, 15139899.987069093j, 27837066.141512733j, 44660155.46798415j, 65724752.28819218j, 91127489.3288743j, 120951688.08989424j, 155270767.32100576j, 194150469.84636205j, 237650401.90699938j, 285825143.95655876j, 338725079.7989731j, 396397032.6674698j, 458884764.3689249j, 526229374.5007638j, 598469624.9733326j, 675642207.5383229j, 757781967.0470047j, 844922089.7807616j, 937094263.8426802j, 1034328816.9251059j, 1136654835.5549896j, 1244100269.0246596j, 1356692020.546476j, 1474456027.6620028j, 1597417333.5462039j, 1725600150.5439067j, 1859027917.0376782j, 1997723348.5574195j, 2141708483.8909173j, 2291004726.8328023j, 2445632884.1102986j, 2605613199.943152j, 2770965387.628281j, 2941708658.4844136j, 3117861748.445721j, 3299442942.5547743j, 3486470097.5725217j, 3678960662.895318j, 3876931699.9455247j, 4080399900.1821365j, 4289381601.86055j, 4503892805.65588j, 4723949189.251216j, 4949566120.981221j, 5180758672.611655j, 5417541631.327026j

In [ ]:
# from scipy.integrate import fixed_quad
# import numpy as np
# from scipy.special import j0
# import matplotlib.pyplot as plt

# b_max = 6
# q_min, q_max = 0, 1

# lst_chi = []
# lst_amp_eik = []

# lst_b_integration = np.linspace(0, b_max, 100)
# lst_sqrt_s = np.linspace(500, 13000, 131)
# db = lst_b_integration[1] - lst_b_integration[0]

# for sqrt_s in lst_sqrt_s:
#     s = sqrt_s**2
#     eik_amp_sum =  0


#     # Loop over b points (so we can store chi(b) values)
#     for b_val in lst_b_integration:

#         # Inner integration over q using fixed_quad
#         def integrand_q(q_val):
#             q2_val = q_val ** 2
#             t = -q2_val

#             diff_t = full_int(
#                 minuit_born.values['mg'],
#                 minuit_born.values['a1'],
#                 minuit_born.values['a2'],
#                 m2_pl,
#                 q2_val,
#                 sqrt_s
#             )

#             born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)
#             return (1/s) * q_val * j0(b_val * q_val) * born_amp

#         chi_val, _ = fixed_quad(integrand_q, q_min, q_max, n=10000)
#         # lst_chi.append(chi_val)
#         factor = 1 - np.exp(1j * chi_val)
#         eik_amp_value = (1j * s) * b_val * j0(b_val * 0) * factor
#         eik_amp_sum += eik_amp_value * db
    
#     lst_amp_eik.append(eik_amp_sum)

#     # Now integrate chi(b) over b also using fixed_quad
#     def integrand_b(b_val):
#         def integrand_q(q_val):
#             q2_val = q_val ** 2
#             t = -q2_val

#             diff_t = full_int(
#                 minuit_born.values['mg'],
#                 minuit_born.values['a1'],
#                 minuit_born.values['a2'],
#                 m2_pl,
#                 q2_val,
#                 sqrt_s
#             )

#             born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)
#             return (1/s) * q_val * j0(b_val * q_val) * born_amp

#         chi_val, _ = fixed_quad(integrand_q, q_min, q_max, n=10000)
#         return chi_val

#     # Full integration over b (for reference or further use)
#     chi_total, _ = fixed_quad(integrand_b, 0, b_max, n=10000)

In [126]:
import numpy as np
from scipy.special import j0

# --- setup parameters ---
step = 200
lst_amp_eik = []

lst_q_integration = np.linspace(0, 1, 50)
lst_b_integration = np.linspace(0, 6, 50)
lst_sqrt_s = np.linspace(500, 13000, 131)

# compute step sizes for the Riemann sum
dq = lst_q_integration[1] - lst_q_integration[0]
db = lst_b_integration[1] - lst_b_integration[0]

for sqrt_s in lst_sqrt_s:
    s = sqrt_s**2

    # initialize amplitude for this sqrt(s)
    eik_amp_sum = 0

    # --- integration over b ---
    for b_val in lst_b_integration:
        chi_sum = 0

        # --- integration over q ---
        for q_val in lst_q_integration:
            # full_int must be your differential amplitude function
            diff_t = full_int(
                ensemble_parameters['atlas']['pl']['mg'],
                ensemble_parameters['atlas']['pl']['a1'],
                ensemble_parameters['atlas']['pl']['a2'],
                m2_pl,
                q_val,
                sqrt_s
            )

            t = -q_val
            born_amp = amp_calculation(
                diff_t,
                s,
                ensemble_parameters['atlas']['pl']['epsilon'],
                t
            )

            chi_val = (1 / s) * (q_val * j0(b_val * q_val)) * born_amp

            # Riemann sum contribution (Δq)
            chi_sum += chi_val * dq

        # after q integration, apply factor and accumulate over b (Δb)
        factor = 1 - np.exp(1j * chi_sum)
        eik_amp_value = (1j * s) * b_val * j0(b_val * 0) * factor
        eik_amp_sum += eik_amp_value * db

    lst_amp_eik.append(eik_amp_sum)
    print(f"√s = {sqrt_s:.0f} GeV -> Eikonal amplitude = {eik_amp_sum}")



√s = 500 GeV -> Eikonal amplitude = 3705513.835759716j
√s = 596 GeV -> Eikonal amplitude = 5291570.3914271j
√s = 692 GeV -> Eikonal amplitude = 7163456.060824985j
√s = 788 GeV -> Eikonal amplitude = 9322206.148568748j
√s = 885 GeV -> Eikonal amplitude = 11768716.870881781j
√s = 981 GeV -> Eikonal amplitude = 14503777.90767808j
√s = 1077 GeV -> Eikonal amplitude = 17528094.723070256j
√s = 1173 GeV -> Eikonal amplitude = 20842304.56949733j
√s = 1269 GeV -> Eikonal amplitude = 24446988.289446697j
√s = 1365 GeV -> Eikonal amplitude = 28342679.38136821j
√s = 1462 GeV -> Eikonal amplitude = 32529871.029414374j
√s = 1558 GeV -> Eikonal amplitude = 37009021.479020104j
√s = 1654 GeV -> Eikonal amplitude = 41780558.69872073j
√s = 1750 GeV -> Eikonal amplitude = 46844884.22864518j
√s = 1846 GeV -> Eikonal amplitude = 52202375.673062734j
√s = 1942 GeV -> Eikonal amplitude = 57853390.327495135j
√s = 2038 GeV -> Eikonal amplitude = 63798265.71143015j
√s = 2135 GeV -> Eikonal amplitude = 70037322.834

In [ ]:
# # integration over sum 1-276-503-334.8683357j
# step = 200

# lst_amp_eik = []

# lst_q_integration = np.linspace(0, 0.2, 50)
# lst_b_integration = np.linspace(0, 30, 50)
# lst_sqrt_s = np.arange(500, 13000, step)
# lst_q_eik = np.linspace(0, 0.1, 50)

# for q_eik_val in lst_q_eik:
    
#     sqrt_s = 13000
#     s = 13000 ** 2

#     # loop sobre os pontos experimentais (para cada t)
#     eik_amp_sum = 0 

#     for b_value in lst_b_integration:
#         chi_sum = 0
        
#         for q_int in lst_q_integration:


#             integral_value = full_int(ensemble_parameters['atlas']['pl']['mg'], 
#                                       ensemble_parameters['atlas']['pl']['a1'], 
#                                       ensemble_parameters['atlas']['pl']['a2'], 
#                                       m2_pl, q_int, sqrt_s)
#             diff_t = integral_value


#             t = -(q_int)

#             born_amp_value = amp_calculation(diff_t, s, ensemble_parameters['atlas']['pl']['epsilon'], t)
#             chi_value = (q_int * j0(b_value * q_int) * born_amp_value) / s

#             chi_sum += chi_value   

#         factor = 1 - np.exp(1j * chi_sum)
#         eik_amp_value = (1j * s) * b_value * j0(b_value * q_eik_val) * factor 
#         eik_amp_sum += eik_amp_value
        
#     lst_amp_eik.append(eik_amp_sum)



In [121]:

fig_amps = go.Figure()

add_total_trace(fig_amps, lst_sqrt_s, np.imag(lst_amp_eik), color='red', label='amp eik', line_style='solid')
add_total_trace(fig_amps, lst_sqrt_s, np.imag(lst_born_amp), color='blue', label='amp born', line_style='solid')


#-----------------------------------------------------------------------------------------------
#----

fig_amps.update_layout(
    title = f'amp vs sqrt - b max = {b_max} - q max = {q_max}',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_amps.update_xaxes(gridcolor='lightgray')
fig_amps.update_yaxes(gridcolor='lightgray')

fig_amps.show(renderer="browser")
fig_amps.write_html(f'../../../../results/eikonal/eik_amp_plots/b_max_{b_max}_q_max_{q_max}')


In [122]:
# import numpy as np
# from scipy.special import j0

# # --- setup parameters ---
# step = 200
# lst_amp_eik_dif = []

# lst_q_integration = np.linspace(0, 0.2, 50)
# lst_b_integration = np.linspace(0, 30, 50)
# # lst_sqrt_s = np.arange(500, 13000, step)
# lst_sqrt_s = np.linspace(500, 13000, 131)
# lst_q_diff_eik = np.linspace(0, 0.1, 50)

# # compute step sizes for the Riemann sum
# dq = lst_q_integration[1] - lst_q_integration[0]
# db = lst_b_integration[1] - lst_b_integration[0]

# for q_diff_eik_val in lst_q_diff_eik:

#     sqrt_s = 13000
#     s = sqrt_s**2

#     # initialize amplitude for this sqrt(s)
#     eik_amp_sum = 0

#     # --- integration over b ---
#     for b_val in lst_b_integration:
#         chi_sum = 0

#         # --- integration over q ---
#         for q_val in lst_q_integration:
#             # full_int must be your differential amplitude function
#             diff_t = full_int(
#                 ensemble_parameters['atlas']['pl']['mg'],
#                 ensemble_parameters['atlas']['pl']['a1'],
#                 ensemble_parameters['atlas']['pl']['a2'],
#                 m2_pl,
#                 q_val,
#                 sqrt_s
#             )

#             t = -q_val
#             born_amp = amp_calculation(
#                 diff_t,
#                 s,
#                 ensemble_parameters['atlas']['pl']['epsilon'],
#                 t
#             )

#             chi_val = (1 / s) * (q_val * j0(b_val * q_val)) * born_amp

#             # Riemann sum contribution (Δq)
#             chi_sum += chi_val * dq

#         # after q integration, apply factor and accumulate over b (Δb)
#         factor = 1 - np.exp(1j * chi_sum)
#         eik_amp_value = (1j * s) * b_val * j0(b_val * q_diff_eik_val) * factor
#         eik_amp_sum += eik_amp_value * db

#     lst_amp_eik_dif.append(eik_amp_sum)
#     print(f"√s = {q_diff_eik_val} GeV -> Eikonal amplitude = {eik_amp_sum}")



In [123]:
# lst_amp_eik_diff_imag = [amp.imag for amp in lst_amp_eik_dif]
# lst_born_amp_diff_imag = [born_amp.imag for born_amp in lst_amp_born_diff]

In [124]:

# fig_amps = go.Figure()

# add_total_trace(fig_amps, lst_q_diff_eik, lst_amp_eik_diff_imag, color='red', label='diff amp eik', line_style='solid')
# add_total_trace(fig_amps, lst_q_diff_eik, lst_born_amp_diff_imag, color='blue', label='diff amp born', line_style='solid')
# #-----------------------------------------------------------------------------------------------
# #----

# fig_amps.update_layout(
#     title='Amp',
#     xaxis_title='|t| (GeV²)',
#     yaxis_title='dσ/dt (mb/GeV²)',
#     yaxis_type='log',
#     legend_title='Mass Model',
#     plot_bgcolor='white',
#     hovermode='x unified'
# )

# fig_amps.update_xaxes(gridcolor='lightgray')
# fig_amps.update_yaxes(gridcolor='lightgray')


# fig_amps.show(renderer="browser")